# Refined: Preparando tabelas e visões para consumo
Para essa etapa já posso começar a pensar em um pipeline analítico. Para centralizar os dados farei uso do duckdb e inclusive subir as tabelas trusted como views e gerando um lineage mais rastrável de dados.

In [1]:
import duckdb
from pathlib import Path

path_db = Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\duckdb\ml_ettj26.duckdb")
con = duckdb.connect(path_db)
con.sql("SHOW TABLES")


┌────────────────────────────────────────────────────────┐
│                          name                          │
│                        varchar                         │
├────────────────────────────────────────────────────────┤
│ vw_refined_b3_forwards_di1                             │
│ vw_refined_b3_swaps_di_pre                             │
│ vw_refined_bcb_demab_government_bonds_secondary_market │
│ vw_refined_bcb_demab_ltn                               │
│ vw_refined_bcb_demab_ntnf                              │
│ vw_refined_calendar_br                                 │
│ vw_refined_ipca                                        │
│ vw_refined_selic                                       │
│ vw_trusted_b3_forwards_di1_lineage                     │
│ vw_trusted_b3_forwards_di1_quotes                      │
│ vw_trusted_b3_forwards_metadata                        │
│ vw_trusted_b3_swaps_dixpre_quotes                      │
│ vw_trusted_b3_swaps_lineage                           

## Calendar: dimensão

Começando pelo mais simples, refinar as tabelas de calendário, sendo elas um dos principais insumos para toda a aplicação, tanto produtos quanto as conveções e curvas. Para todos os efeitos temos duas conveções principais: Dias Corridos e Dias Úteis, sendo aqui a primeira decisão a se tomar para fins de velocidade e praticidade: Ao invés de realizar operações com datetime e cálculos com fins de semana e feriado toda vez que for precificar um produto ou calcular um fator, criei uma formatação de índice, sendo assim otimizando a operação com valores inteiros e qualquer distância entre dadtas se torna uma subtração ao invés de uma busca.

Outro tipo de coluna interessante além das data e índices, seriam flags indicando se é dia útil, feriado ou fim de semana, facilitando filtros com booleanos. Já pensando em views, agregações e análises, também colunas que evitem recálculos extensivos com ano, mês dia do mês e dia da semana serão extremamente úteis agilizando trabalho quando construindo dashboards e definindo regras de negócio, tomando por exemplo produtos que seguem a IMM por exemplo que vencem sempre no dia útil mais próximo do 20 de meses específicos, ou produtos que vencem sempre na segunda quarta do mês.

outras colunas de suporte como lineage da fonte que facilitam o processo de governança da informação e nome de feriado que é mais usado para o cálculo interno das outras colunas também será mantido.

In [2]:
import pandas as pd
holidays = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\calendars\02_trusted\ref\anbima_holidays.parquet")
calendar_bd = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\calendars\02_trusted\ref\calendar_bd_index.parquet")

print(holidays.info())
holidays.head()

print(calendar_bd.info())
calendar_bd.head()

<class 'pandas.DataFrame'>
RangeIndex: 1263 entries, 0 to 1262
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   cal_id            1263 non-null   str                
 1   date              1263 non-null   datetime64[ms, UTC]
 2   holiday_name      1263 non-null   str                
 3   weekday           1263 non-null   int32              
 4   ingestion_ts_utc  1263 non-null   str                
 5   source_file_hash  1263 non-null   str                
 6   pipeline_run_id   1263 non-null   str                
dtypes: datetime64[ms, UTC](1), int32(1), str(5)
memory usage: 214.4 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 36159 entries, 0 to 36158
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   cal_id            36159 non-null  str                
 1   date              

,cal_id,date,weekday,is_business_day,bd_index,holiday_name,ingestion_ts_utc,source_file_hash,pipeline_run_id
0,BR_ANBIMA,2001-01-01 00:00:00+00:00,0,False,0,Confraternização Universal,2026-02-23T01:13:13+00:00,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...,local
1,BR_ANBIMA,2001-01-02 00:00:00+00:00,1,True,1,NaN,2026-02-23T01:13:13+00:00,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...,local
2,BR_ANBIMA,2001-01-03 00:00:00+00:00,2,True,2,NaN,2026-02-23T01:13:13+00:00,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...,local
3,BR_ANBIMA,2001-01-04 00:00:00+00:00,3,True,3,NaN,2026-02-23T01:13:13+00:00,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...,local
4,BR_ANBIMA,2001-01-05 00:00:00+00:00,4,True,4,NaN,2026-02-23T01:13:13+00:00,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...,local


A partir disso e tendo os dois principais arquivos de fonte (lista de feriados ANBIMA e uma trusted de Calendário) podemos ver que o arquivo de calendário é a trusted já num formato semi pronto para consumo, precisando de apenas alguns tratamentos e cáculos para estar pronto para consumo

In [16]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\03_refined\calendars\Calendar_BR_dim.parquet")
print(df.info())
df.head(7)

<class 'pandas.DataFrame'>
RangeIndex: 36159 entries, 0 to 36158
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   calendar_id       36159 non-null  str           
 1   date              36159 non-null  datetime64[us]
 2   year              36159 non-null  int32         
 3   month             36159 non-null  int32         
 4   day               36159 non-null  int32         
 5   weekday           36159 non-null  int32         
 6   is_weekend        36159 non-null  bool          
 7   is_holiday        36159 non-null  bool          
 8   is_business_day   36159 non-null  bool          
 9   act_index         36159 non-null  int64         
 10  bd_index          36159 non-null  int64         
 11  holiday_name      1263 non-null   str           
 12  source_file_hash  36159 non-null  str           
dtypes: bool(3), datetime64[us](1), int32(4), int64(2), str(3)
memory usage: 4.9 MB
None


,calendar_id,date,year,month,day,weekday,is_weekend,is_holiday,is_business_day,act_index,bd_index,holiday_name,source_file_hash
0,BR_ANBIMA,2001-01-01,2001,1,1,0,False,True,False,0,0,Confraternização Universal,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
1,BR_ANBIMA,2001-01-02,2001,1,2,1,False,False,True,1,1,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
2,BR_ANBIMA,2001-01-03,2001,1,3,2,False,False,True,2,2,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
3,BR_ANBIMA,2001-01-04,2001,1,4,3,False,False,True,3,3,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
4,BR_ANBIMA,2001-01-05,2001,1,5,4,False,False,True,4,4,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
5,BR_ANBIMA,2001-01-06,2001,1,6,5,True,False,False,5,4,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
6,BR_ANBIMA,2001-01-07,2001,1,7,6,True,False,False,6,4,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...


## BCB data: Dados Históricos do BACEN

Partindo para proxima etapa temos dois objetos para estruturar:
- Dados históricos da SELIC provenientes da SGS
- Dados históricos dos títulos públicos negociados pelo DEMAB



### SGS

Os dados históricos da taxa SELIC e índice de inflação IPCA são os mais simples de desenvolver as views por já estarem bem estruturados, sendo nescessário apenas um merge e filtro.

In [26]:
df_sgs_meta = pd.read_parquet(Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\bcb\sgs\series_meta.parquet"))
df_sgs_points = pd.read_parquet(Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\bcb\sgs\points.parquet"))

print(df_sgs_meta.info())
print(df_sgs_points.info())
df_sgs_meta

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   series_id    2 non-null      int64
 1   series_name  2 non-null      str  
 2   frequency    2 non-null      str  
 3   unit         2 non-null      str  
 4   source       2 non-null      str  
dtypes: int64(1), str(4)
memory usage: 244.0 bytes
None
<class 'pandas.DataFrame'>
RangeIndex: 9852 entries, 0 to 9851
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   series_id         9852 non-null   int64  
 1   ref_date          9852 non-null   object 
 2   value             9852 non-null   float64
 3   record_hash       9852 non-null   str    
 4   raw_file          9852 non-null   str    
 5   raw_hash          9852 non-null   str    
 6   ingestion_ts_utc  9852 non-null   str    
dtypes: float64(1), int64(1), object(1), str(4)
memory us

,series_id,series_name,frequency,unit,source
0,432,SELIC,D,% a.a.,BCB_SGS
1,433,IPCA,M,%,BCB_SGS


In [3]:
import pandas as pd

path_refined_selic = Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\03_refined\bcb\sgs\selic.parquet")
path_refined_ipca = Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\03_refined\bcb\sgs\ipca.parquet")

df_selic = pd.read_parquet(path_refined_selic)
df_ipca = pd.read_parquet(path_refined_ipca)   

df_selic

,date,series_id,series_name,value,unit,frequency,source,raw_hash
0,2000-01-01,432,SELIC,19.0,% a.a.,D,BCB_SGS,023c3e8e15126a84ae1b8a9e00a325719c03eb7c8f8bbb...
1,2000-01-02,432,SELIC,19.0,% a.a.,D,BCB_SGS,023c3e8e15126a84ae1b8a9e00a325719c03eb7c8f8bbb...
2,2000-01-03,432,SELIC,19.0,% a.a.,D,BCB_SGS,023c3e8e15126a84ae1b8a9e00a325719c03eb7c8f8bbb...
3,2000-01-04,432,SELIC,19.0,% a.a.,D,BCB_SGS,023c3e8e15126a84ae1b8a9e00a325719c03eb7c8f8bbb...
4,2000-01-05,432,SELIC,19.0,% a.a.,D,BCB_SGS,023c3e8e15126a84ae1b8a9e00a325719c03eb7c8f8bbb...
...,...,...,...,...,...,...,...,...
9534,2026-02-07,432,SELIC,15.0,% a.a.,D,BCB_SGS,480c424330ea60ca1d9e880bb515e8be38bb0b61f0c3d6...
9535,2026-02-08,432,SELIC,15.0,% a.a.,D,BCB_SGS,480c424330ea60ca1d9e880bb515e8be38bb0b61f0c3d6...
9536,2026-02-09,432,SELIC,15.0,% a.a.,D,BCB_SGS,480c424330ea60ca1d9e880bb515e8be38bb0b61f0c3d6...
9537,2026-02-10,432,SELIC,15.0,% a.a.,D,BCB_SGS,480c424330ea60ca1d9e880bb515e8be38bb0b61f0c3d6...


### DEMAB
Já para os dados da DEMAB temos a complexidade de muitos dados diários, porém o duckdb consegue lidar muito bem com isso sem a nescessidade de listar todos os arquivos do diretório ou outras técnicas de mais baixo nível. 

In [5]:
import os
path_data = "C:/Users/Dell/OneDrive/Documentos/GitHub/ML-ETTJ26/data"
path_demab = os.path.join(path_data, "02_trusted", "bcb", "demab")
path_demab_quotes = os.path.join(path_demab,  "quotes_daily", "*.parquet" )
path_demab_masters = os.path.join(path_demab,  "instruments.parquet" )

df_demab_ref = con.execute(f"""
    SELECT *
    FROM read_parquet('{path_demab_quotes}') AS quotes
    """).df()
df_demab_ref.info()

<class 'pandas.DataFrame'>
RangeIndex: 197738 entries, 0 to 197737
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   trade_date        197738 non-null  datetime64[us]
 1   isin              197738 non-null  str           
 2   pu_min            187169 non-null  float64       
 3   pu_med            187169 non-null  float64       
 4   pu_max            187169 non-null  float64       
 5   pu_lastro         197738 non-null  float64       
 6   valor_par         197522 non-null  float64       
 7   taxa_min          102480 non-null  float64       
 8   taxa_med          102480 non-null  float64       
 9   taxa_max          102478 non-null  float64       
 10  raw_zip_file      197738 non-null  str           
 11  raw_zip_hash      197738 non-null  str           
 12  inner_file        197738 non-null  str           
 13  record_hash       197738 non-null  str           
 14  ingestion_ts_ut

Com essa pequena análise inicial já pode-se perceber que temos problemas de missing data no nosso dataframe que pode ser um empecilho para futuras etapas do projeto. Com sorte a coluna de pu_lastro e isin não faltam dados e como os títulos LTN e NTN-F que serão usados possuem valor par e calendário padronizados pode-se reconstruir suas taxas caso não estejam preenchidas. Com isso, vale construir já de colunas de flag para que nas proximas tabelas refined downstream posso com maior facilidade identificar essas observações para tratamente (usando filtro ao inves de operações com múltiplas colunas)

In [6]:
import pandas as pd
df_demab_masters = pd.read_parquet(path_demab_masters)
print(df_demab_masters.info())
df_demab_masters["sigla"].unique()

<class 'pandas.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   isin             407 non-null    str   
 1   sigla            407 non-null    str   
 2   emissao_date     407 non-null    object
 3   vencimento_date  407 non-null    object
 4   source           407 non-null    str   
dtypes: object(2), str(3)
memory usage: 26.0+ KB
None


<ArrowStringArray>
[   'LTN', 'BTNBIB',    'LFT',  'LFT-A',  'LFT-B', 'NTN-A3', 'NTN-A6',
  'NTN-B',  'NTN-C',  'NTN-D',  'NTN-F']
Length: 11, dtype: str

In [7]:
import pandas as pd
path_demab_refined = r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\03_refined\bcb\demab\demab_government_bonds_secondary_market_refined.parquet"
demab_refined = pd.read_parquet(path_demab_refined)
print(demab_refined.info())
demab_refined[demab_refined["sigla"]=="LTN"].tail()

<class 'pandas.DataFrame'>
RangeIndex: 197738 entries, 0 to 197737
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   date              197738 non-null  datetime64[us]
 1   isin              197738 non-null  str           
 2   sigla             197738 non-null  str           
 3   pu_lastro         197738 non-null  float64       
 4   valor_par         197522 non-null  float64       
 5   issue_date        197738 non-null  datetime64[us]
 6   maturity          197738 non-null  datetime64[us]
 7   days_to_maturity  197738 non-null  int64         
 8   bd_to_maturity    197738 non-null  int64         
 9   pu_min            187169 non-null  float64       
 10  pu_med            187169 non-null  float64       
 11  pu_max            187169 non-null  float64       
 12  taxa_min          102480 non-null  float64       
 13  taxa_med          102480 non-null  float64       
 14  taxa_max       

,date,isin,sigla,pu_lastro,valor_par,issue_date,maturity,days_to_maturity,bd_to_maturity,pu_min,pu_med,pu_max,taxa_min,taxa_med,taxa_max,source,raw_hash
197696,2026-01-30,BRSTNCLTN8F6,LTN,741.226659,1000.0,2024-07-05,2028-07-01,883,603,749.181957,749.960816,751.495173,12.6590,12.7551,12.8040,BCB_DEMAB,1d33f2b9031c7f6ffd208a26e6b1e9812b48d21fcdec0f...
197700,2026-01-30,BRSTNCLTN806,LTN,696.218146,1000.0,2022-02-11,2029-01-01,1067,728,704.487149,706.290808,706.808868,12.7625,12.7911,12.8910,BCB_DEMAB,1d33f2b9031c7f6ffd208a26e6b1e9812b48d21fcdec0f...
197705,2026-01-30,BRSTNCLTN8K6,LTN,652.223439,1000.0,2025-07-04,2029-07-01,1248,852,661.304354,663.103830,663.783390,12.8860,12.9202,13.0110,BCB_DEMAB,1d33f2b9031c7f6ffd208a26e6b1e9812b48d21fcdec0f...
197707,2026-01-30,BRSTNCLTN8A7,LTN,610.477537,1000.0,2024-01-05,2030-01-01,1432,977,615.843022,622.001018,622.823123,12.9900,13.0285,13.3189,BCB_DEMAB,1d33f2b9031c7f6ffd208a26e6b1e9812b48d21fcdec0f...
197719,2026-01-30,BRSTNCLTN8J8,LTN,465.051123,1000.0,2025-01-10,2032-01-01,2162,1481,475.302503,478.752849,479.931269,13.3050,13.3524,13.4920,BCB_DEMAB,1d33f2b9031c7f6ffd208a26e6b1e9812b48d21fcdec0f...


In [6]:
df_ntnf = con.sql("SELECT * FROM vw_refined_bcb_demab_ntnf ").df()
df_ltn = con.sql("SELECT * FROM vw_refined_bcb_demab_ltn ").df()
print(df_ntnf.info())
df_ntnf.tail()


<class 'pandas.DataFrame'>
RangeIndex: 26900 entries, 0 to 26899
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              26900 non-null  datetime64[us]
 1   isin              26900 non-null  str           
 2   sigla             26900 non-null  str           
 3   pu_lastro         26900 non-null  float64       
 4   valor_par         26897 non-null  float64       
 5   issue_date        26900 non-null  datetime64[us]
 6   maturity          26900 non-null  datetime64[us]
 7   days_to_maturity  26900 non-null  int64         
 8   bd_to_maturity    26900 non-null  int64         
 9   pu_min            25805 non-null  float64       
 10  pu_med            25805 non-null  float64       
 11  pu_max            25805 non-null  float64       
 12  taxa_min          0 non-null      float64       
 13  taxa_med          0 non-null      float64       
 14  taxa_max          0 non-null     

,date,isin,sigla,pu_lastro,valor_par,issue_date,maturity,days_to_maturity,bd_to_maturity,pu_min,pu_med,pu_max,taxa_min,taxa_med,taxa_max,source,raw_hash
26895,2026-01-30,BRSTNCNTF1Q6,NTN-F,935.065302,1000.0,2018-01-05,2029-01-01,1067,728,942.076227,946.217571,947.775432,NaN,NaN,NaN,BCB_DEMAB,1d33f2b9031c7f6ffd208a26e6b1e9812b48d21fcdec0f...
26896,2026-01-30,BRSTNCNTF204,NTN-F,887.638628,1000.0,2020-01-10,2031-01-01,1797,1229,896.251624,901.733691,902.969608,NaN,NaN,NaN,BCB_DEMAB,1d33f2b9031c7f6ffd208a26e6b1e9812b48d21fcdec0f...
26897,2026-01-30,BRSTNCNTF212,NTN-F,847.865228,1000.0,2022-01-07,2033-01-01,2528,1732,860.408543,863.274974,866.868320,NaN,NaN,NaN,BCB_DEMAB,1d33f2b9031c7f6ffd208a26e6b1e9812b48d21fcdec0f...
26898,2026-01-30,BRSTNCNTF238,NTN-F,819.454734,1000.0,2024-01-05,2035-01-01,3258,2232,838.697985,841.873478,844.894712,NaN,NaN,NaN,BCB_DEMAB,1d33f2b9031c7f6ffd208a26e6b1e9812b48d21fcdec0f...
26899,2026-01-30,BRSTNCNTF2K7,NTN-F,796.135217,1000.0,2026-01-09,2037-01-01,3989,2734,816.592865,821.727840,823.755937,NaN,NaN,NaN,BCB_DEMAB,1d33f2b9031c7f6ffd208a26e6b1e9812b48d21fcdec0f...


In [5]:
df_ntnf["valor_par"].describe()

count    26897.0
mean      1000.0
std          0.0
min       1000.0
25%       1000.0
50%       1000.0
75%       1000.0
max       1000.0
Name: valor_par, dtype: float64

In [4]:
print(df_ltn.info())

<class 'pandas.DataFrame'>
RangeIndex: 46348 entries, 0 to 46347
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              46348 non-null  datetime64[us]
 1   isin              46348 non-null  str           
 2   sigla             46348 non-null  str           
 3   pu_lastro         46348 non-null  float64       
 4   valor_par         46348 non-null  float64       
 5   issue_date        46348 non-null  datetime64[us]
 6   maturity          46348 non-null  datetime64[us]
 7   days_to_maturity  46348 non-null  int64         
 8   bd_to_maturity    46348 non-null  int64         
 9   pu_min            43649 non-null  float64       
 10  pu_med            43649 non-null  float64       
 11  pu_max            43649 non-null  float64       
 12  taxa_min          43649 non-null  float64       
 13  taxa_med          43649 non-null  float64       
 14  taxa_max          43649 non-null 

## B3


### Forwards de DI1

In [10]:
import pandas as pd

path_quotes = r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\b3\forwards\di1_quotes_daily\*.parquet"

df_master = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\b3\forwards\di1_instrument_master.parquet")
df_lineage = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\b3\forwards\di1_lineage\2020-01-07.parquet")
df_points = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\b3\forwards\di1_quotes_daily\2020-01-07.parquet")


df_di1_ref = con.execute(f"""
    SELECT *
    FROM read_parquet('{path_quotes}') AS quotes
    """).df()

print(df_master.info())
print(df_lineage.info())

<class 'pandas.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   TckrSymb             122 non-null    str                
 1   asset                122 non-null    str                
 2   contract_month_code  122 non-null    str                
 3   contract_year        122 non-null    int64              
 4   maturity_date        122 non-null    datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), int64(1), str(3)
memory usage: 6.1 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   lineage_id        1 non-null      str                
 1   outer_zip         1 non-null      str                
 2   inner_zip         1 non-null      str                
 3   xml_name 

In [11]:
print(df_di1_ref.info())
df_points.head()

<class 'pandas.DataFrame'>
RangeIndex: 58474 entries, 0 to 58473
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype                            
---  ------            --------------  -----                            
 0   TradDt            58474 non-null  datetime64[us, America/Sao_Paulo]
 1   TckrSymb          58474 non-null  str                              
 2   snapshot_ts_utc   58474 non-null  datetime64[us, America/Sao_Paulo]
 3   AdjstdQtTax       58473 non-null  float64                          
 4   AdjstdQt          58473 non-null  float64                          
 5   BestBidPric       39606 non-null  float64                          
 6   BestAskPric       39215 non-null  float64                          
 7   LastPric          52046 non-null  float64                          
 8   TradAvrgPric      52046 non-null  float64                          
 9   MinPric           52046 non-null  float64                          
 10  MaxPric           520

,TradDt,TckrSymb,snapshot_ts_utc,AdjstdQtTax,AdjstdQt,BestBidPric,BestAskPric,LastPric,TradAvrgPric,MinPric,MaxPric,TradQty,FinInstrmQty,OpnIntrst,lineage_id,ingestion_ts_utc
109,2020-01-07 00:00:00+00:00,DI1G20,2020-01-07 22:40:57+00:00,4.402,99675.73,4.402,4.403,4.402,4.402,4.402,4.405,24.0,41410.0,1021286.0,PR200107_20200107.zip|<inner_in_memory.zip>|BV...,2026-04-02 16:48:14.973829+00:00
110,2020-01-07 00:00:00+00:00,DI1U20,2020-01-07 22:40:57+00:00,4.350,97266.93,4.305,4.340,4.350,4.350,4.350,4.350,2.0,400.0,35510.0,PR200107_20200107.zip|<inner_in_memory.zip>|BV...,2026-04-02 16:48:14.973829+00:00
111,2020-01-07 00:00:00+00:00,DI1N23,2020-01-07 22:40:57+00:00,6.000,81702.03,5.970,5.990,5.980,6.016,5.980,6.110,628.0,19910.0,766330.0,PR200107_20200107.zip|<inner_in_memory.zip>|BV...,2026-04-02 16:48:14.973829+00:00
112,2020-01-07 00:00:00+00:00,DI1J20,2020-01-07 22:40:57+00:00,4.315,99015.81,4.315,4.316,4.315,4.329,4.306,4.337,750.0,367715.0,4677014.0,PR200107_20200107.zip|<inner_in_memory.zip>|BV...,2026-04-02 16:48:14.973829+00:00
113,2020-01-07 00:00:00+00:00,DI1F25,2020-01-07 22:40:57+00:00,6.440,73321.03,6.420,6.430,6.430,6.477,6.420,6.550,3643.0,110095.0,963142.0,PR200107_20200107.zip|<inner_in_memory.zip>|BV...,2026-04-02 16:48:14.973829+00:00


In [4]:
df_master.head()

,TckrSymb,asset,contract_month_code,contract_year,maturity_date
0,DI1G27,DI1,G,2027,2027-02-01 00:00:00+00:00
1,DI1X29,DI1,X,2029,2029-11-01 00:00:00+00:00
2,DI1X28,DI1,X,2028,2028-11-01 00:00:00+00:00
3,DI1Z27,DI1,Z,2027,2027-12-01 00:00:00+00:00
4,DI1Z29,DI1,Z,2029,2029-12-03 00:00:00+00:00


In [3]:
import pandas as pd
df_out = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\03_refined\b3\forwards\b3_forwards_di1_refined.parquet")
print(df_out.info())
df_out.tail()

<class 'pandas.DataFrame'>
RangeIndex: 58474 entries, 0 to 58473
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              58474 non-null  datetime64[us]
 1   asset             58474 non-null  str           
 2   ticker            58474 non-null  str           
 3   adjusted_price    58473 non-null  float64       
 4   adjusted_pu       58473 non-null  float64       
 5   maturity          58474 non-null  datetime64[us]
 6   days_to_maturity  58474 non-null  int64         
 7   bd_to_maturity    58474 non-null  int64         
 8   bid_price         39606 non-null  float64       
 9   ask_price         39215 non-null  float64       
 10  close_price       52046 non-null  float64       
 11  average_price     52046 non-null  float64       
 12  minimum_price     52046 non-null  float64       
 13  maximum_price     52046 non-null  float64       
 14  quantity          52046 non-null 

,date,asset,ticker,adjusted_price,adjusted_pu,maturity,days_to_maturity,bd_to_maturity,bid_price,ask_price,close_price,average_price,minimum_price,maximum_price,quantity
58469,2026-02-26,DI1,DI1F37,13.366,25869.99,2037-01-01,3962,2716,13.370,13.38,13.380,13.364,13.330,13.48,5512.0
58470,2026-02-26,DI1,DI1F38,13.338,22920.51,2038-01-03,4329,2965,13.325,NaN,13.345,13.329,13.300,13.40,353.0
58471,2026-02-26,DI1,DI1F39,13.320,20274.25,2039-01-02,4693,3216,NaN,NaN,13.365,13.367,13.365,13.37,2.0
58472,2026-02-26,DI1,DI1F40,13.303,17937.01,2040-01-01,5057,3467,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58473,2026-02-26,DI1,DI1F41,13.310,15832.28,2041-01-01,5423,3717,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Swap DI x PRE


In [7]:
import pandas as pd

df_lineage = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\b3\swaps\data_lineage.parquet")
df_master = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\b3\swaps\swap_master.parquet")

path_points = r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\b3\swaps\swap_dixpre\*.parquet"


df_swaps = con.execute(f"""
    SELECT *
    FROM read_parquet('{path_points}') AS quotes
    """).df()

In [8]:
print(df_master.info())
print(df_lineage.info())
df_master

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 30 to 30
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   CodProd     1 non-null      str  
 1   nome        1 non-null      str  
 2   underlying  1 non-null      str  
 3   fixed_leg   1 non-null      str  
 4   float_leg   1 non-null      str  
 5   calendar    1 non-null      str  
dtypes: str(6)
memory usage: 210.0 bytes
None
<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   lineage_id  31 non-null     str  
 1   outer_zip   31 non-null     str  
 2   inner_zip   31 non-null     str  
 3   txt_name    31 non-null     str  
 4   hash_file   31 non-null     str  
dtypes: str(5)
memory usage: 6.6 KB
None


,CodProd,nome,underlying,fixed_leg,float_leg,calendar
30,T1PRE,DIxPRE,DI x PRE,PRE,DI,BU/252


In [8]:
print(df_swaps.info())
df_swaps.tail()

<class 'pandas.DataFrame'>
RangeIndex: 412297 entries, 0 to 412296
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype                            
---  ------            --------------   -----                            
 0   TradDt            412297 non-null  datetime64[us]                   
 1   CodProd           412297 non-null  str                              
 2   days_to_maturity  412297 non-null  int64                            
 3   bd_to_maturity    412297 non-null  int64                            
 4   days_to_delivery  412297 non-null  int64                            
 5   raw_value         412297 non-null  int64                            
 6   raw_signal        412297 non-null  str                              
 7   adjusted_value    412297 non-null  float64                          
 8   tipo_cotacao      412297 non-null  str                              
 9   lineage_id        412297 non-null  str                              
 10  ingesti

,TradDt,CodProd,days_to_maturity,bd_to_maturity,days_to_delivery,raw_value,raw_signal,adjusted_value,tipo_cotacao,lineage_id,ingestion_ts_utc
412292,2026-02-13,T1PRE,8949,6134,8949,134300000,+,1343.0,M,aac06e2ae394757fbf2e278ec030506cfeac99ba181ef2...,2026-04-01 23:46:23.336701-03:00
412293,2026-02-13,T1PRE,9000,6170,9000,134310000,+,1343.1,F,aac06e2ae394757fbf2e278ec030506cfeac99ba181ef2...,2026-04-01 23:46:23.336701-03:00
412294,2026-02-13,T1PRE,10685,7323,10685,134400000,+,1344.0,M,aac06e2ae394757fbf2e278ec030506cfeac99ba181ef2...,2026-04-01 23:46:23.336701-03:00
412295,2026-02-13,T1PRE,10800,7404,10800,134400000,+,1344.0,F,aac06e2ae394757fbf2e278ec030506cfeac99ba181ef2...,2026-04-01 23:46:23.336701-03:00
412296,2026-02-13,T1PRE,12603,8640,12603,134470000,+,1344.7,M,aac06e2ae394757fbf2e278ec030506cfeac99ba181ef2...,2026-04-01 23:46:23.337701-03:00


In [ ]:
import pandas as pd
df_swaps_refined = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\03_refined\b3\swaps\b3_swaps_dipre_refined.parquet")
print(df_swaps_refined.info())
df_swaps_refined.head()

<class 'pandas.DataFrame'>
RangeIndex: 412297 entries, 0 to 412296
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   date              412297 non-null  datetime64[us]
 1   product_code      412297 non-null  str           
 2   name              412297 non-null  str           
 3   underlying        412297 non-null  str           
 4   adjusted_value    412297 non-null  float64       
 5   maturity          412297 non-null  datetime64[us]
 6   days_to_maturity  412297 non-null  int64         
 7   bd_to_maturity    412297 non-null  int64         
 8   days_to_delivery  412297 non-null  int64         
 9   fixed_leg         412297 non-null  str           
 10  float_leg         412297 non-null  str           
 11  quote_type        412297 non-null  str           
 12  lineage_id        412297 non-null  str           
dtypes: datetime64[us](2), float64(1), int64(3), str(7)
memory usage: 75.9 

,date,product_code,name,underlying,adjusted_value,maturity,days_to_maturity,bd_to_maturity,days_to_delivery,fixed_leg,float_leg,quote_type,lineage_id
412292,2026-02-13,T1PRE,DIxPRE,DI x PRE,13.430,2050-08-15,8949,6134,8949,PRE,DI,M,aac06e2ae394757fbf2e278ec030506cfeac99ba181ef2...
412293,2026-02-13,T1PRE,DIxPRE,DI x PRE,13.431,2050-10-05,9000,6170,9000,PRE,DI,F,aac06e2ae394757fbf2e278ec030506cfeac99ba181ef2...
412294,2026-02-13,T1PRE,DIxPRE,DI x PRE,13.440,2055-05-17,10685,7323,10685,PRE,DI,M,aac06e2ae394757fbf2e278ec030506cfeac99ba181ef2...
412295,2026-02-13,T1PRE,DIxPRE,DI x PRE,13.440,2055-09-09,10800,7404,10800,PRE,DI,F,aac06e2ae394757fbf2e278ec030506cfeac99ba181ef2...
412296,2026-02-13,T1PRE,DIxPRE,DI x PRE,13.447,2060-08-16,12603,8640,12603,PRE,DI,M,aac06e2ae394757fbf2e278ec030506cfeac99ba181ef2...
